This notebook attempts to load existing trained siamese backbones, add a new classification head, then attempt to tune for classification.

In [1]:
# Import helper code
import sys
sys.path.insert(1, '../')
import helpers
import torch

In [15]:
import os
dataset_path = "../split_node21_sets/"
process =  "lung_seg" #"crop"  "arch_seg"
train_set = "chestxray14"
bsz = 64
resize_dim = 224
# run_index = 0
# best_epoch_to_load = 97
# run_index = 1
# best_epoch_to_load = 98
run_index = 2
best_epoch_to_load = 50
weights = torch.load(f'logs/subsets/chestxray14/rad_unfrz_cosine_{process}_{run_index}/checkpoints/checkpoint_epoch_{best_epoch_to_load}.pth')
# weights = torch.load('logs/subsets/chestxray14/rad_unfrz_cosine_crop_0/checkpoints/best_model.pth',map_location=device)

from torchvision import transforms
import torch
base_transform = transforms.Compose([
    transforms.Resize((resize_dim, resize_dim)),
    transforms.ToTensor(),
])

augment_transform = transforms.Compose([
    transforms.Resize((resize_dim, resize_dim)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])
# Load with default settings
train_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,train_set, "train"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=augment_transform,
    cache_in_ram=True

)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  4.35it/s]


Caching base images into RAM...


Caching: 100%|██████████| 2478/2478 [00:13<00:00, 181.09it/s]

Cached 2478 images
Total pairs: 1239
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 420
  normal (idx=0): 819


In [16]:
test_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,train_set, "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  9.38it/s]


Caching base images into RAM...


Caching: 100%|██████████| 1062/1062 [00:06<00:00, 173.58it/s]

Cached 1062 images
Total pairs: 531
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 180
  normal (idx=0): 351


In [17]:
test2_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,"padchest", "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 10.73it/s]


Caching base images into RAM...


Caching: 100%|██████████| 1008/1008 [00:05<00:00, 172.18it/s]

Cached 1008 images
Total pairs: 504
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 94
  normal (idx=0): 410


In [18]:
test3_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,"jsrt", "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 66.34it/s]


Caching base images into RAM...


Caching: 100%|██████████| 144/144 [00:00<00:00, 174.36it/s]

Cached 144 images
Total pairs: 72
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 44
  normal (idx=0): 28


## Create Classifier Head

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SiameseClassifier(nn.Module):
    """
    Wraps trained SiameseNetwork with a classification head
    Default freeze the Siamese backbone for fast fine-tuning
    """
    def __init__(self, siamese_model, embedding_dim=128, freeze_siamese=True):
        super(SiameseClassifier, self).__init__()
        
        self.siamese = siamese_model
        
        # Freeze the siamese network if desired
        if freeze_siamese:
            for param in self.siamese.parameters():
                param.requires_grad = False
        
        # Classification head operates on distance/concatenated embeddings
        # Use both distance + embeddings
        
        input_dim = 2 * embedding_dim + 1  # concatenated embeddings + distance
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)  # Binary classification (sigmoid applied later)
        )
        
    def forward(self, x1, x2, return_embeddings=False, distance_metric='cosine'):
        """
        Args:
            x1, x2: Input image pairs
            return_embeddings: If True, also return embeddings for analysis
            distance_metric: 'euclidean' or 'cosine'
        Returns:
            logits: Classification logits (before sigmoid)
            (optional) emb1, emb2, distance
        """
        # Get embeddings from siamese network
        emb1, emb2 = self.siamese(x1, x2)
        
        # Calculate distance
        if distance_metric == 'euclidean':
            distance = F.pairwise_distance(emb1, emb2, p=2)
        elif distance_metric == 'cosine':
            cosine_sim = torch.sum(emb1 * emb2, dim=1)
            distance = 1 - cosine_sim
        else:
            raise ValueError(f"Unknown distance metric: {distance_metric}")
        
        # Concatenate embeddings and distance
        # Shape: (batch, 2*embedding_dim + 1)
        combined = torch.cat([emb1, emb2, distance.unsqueeze(1)], dim=1)
        
        # Get classification logits
        logits = self.classifier(combined).squeeze()
        
        if return_embeddings:
            return logits, emb1, emb2, distance
        return logits

In [25]:
# import training metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

In [26]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, freeze_backbone=True).to(device)

# contrastive_loss = helpers.losses.ContrastiveLoss(margin=1.0)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.load_state_dict(weights['model_state_dict'])

classifier = SiameseClassifier(model, embedding_dim=128, freeze_siamese=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

In [22]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, freeze_backbone=True).to(device)

# contrastive_loss = helpers.losses.ContrastiveLoss(margin=1.0)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.load_state_dict(weights['model_state_dict'])

classifier = SiameseClassifier(model, embedding_dim=128, freeze_siamese=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

In [27]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'rad_cxr14_cosine_{process}_{run_index}_best_classifier_f1_1.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5644 | Acc: 0.8410 | Prec: 0.9912 | Rec: 0.5357 | F1: 0.6955 | AUC: 0.9592
  Test   Loss: 0.5035 | Acc: 0.7815 | Prec: 0.8404 | Rec: 0.4389 | F1: 0.5766 | AUC: 0.7991
  Padchest   Loss: 0.4228 | Acc: 0.8552 | Prec: 0.7234 | Rec: 0.3617 | F1: 0.4823 | AUC: 0.7621
  JSRT   Loss: 0.8206 | Acc: 0.5278 | Prec: 0.9167 | Rec: 0.2500 | F1: 0.3929 | AUC: 0.6883
  ✓ Saved new best model! (F1: 0.5766)
Epoch 2/20
  Train Loss: 0.2145 | Acc: 0.9483 | Prec: 0.9734 | Rec: 0.8714 | F1: 0.9196 | AUC: 0.9827
  Test   Loss: 0.5980 | Acc: 0.8023 | Prec: 0.8378 | Rec: 0.5167 | F1: 0.6392 | AUC: 0.8030
  Padchest   Loss: 0.4609 | Acc: 0.8552 | Prec: 0.6721 | Rec: 0.4362 | F1: 0.5290 | AUC: 0.7544
  JSRT   Loss: 1.6817 | Acc: 0.5694 | Prec: 0.9333 | Rec: 0.3182 | F1: 0.4746 | AUC: 0.6981
  ✓ Saved new best model! (F1: 0.6392)
Epoch 3/20
  Train Loss: 0.1014 | Acc: 0.9645 | Prec: 0.9434 | Rec: 0.9524 | F1: 0.9479 | AUC: 0.9905
  Test   Loss: 0.6941 | Acc: 0.8060 | Prec: 0.8130 | Rec

In [28]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, freeze_backbone=True).to(device)

# contrastive_loss = helpers.losses.ContrastiveLoss(margin=1.0)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.load_state_dict(weights['model_state_dict'])

classifier = SiameseClassifier(model, embedding_dim=128, freeze_siamese=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [29]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'rad_cxr14_cosine_{process}_{run_index}_best_classifier_f1_2.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5938 | Acc: 0.8822 | Prec: 0.7665 | Rec: 0.9381 | F1: 0.8437 | AUC: 0.9599
  Test   Loss: 0.5205 | Acc: 0.8004 | Prec: 0.8364 | Rec: 0.5111 | F1: 0.6345 | AUC: 0.7991
  Padchest   Loss: 0.4704 | Acc: 0.8571 | Prec: 0.6833 | Rec: 0.4362 | F1: 0.5325 | AUC: 0.7627
  JSRT   Loss: 0.8017 | Acc: 0.5417 | Prec: 0.9231 | Rec: 0.2727 | F1: 0.4211 | AUC: 0.6867
  ✓ Saved new best model! (F1: 0.6345)
Epoch 2/20
  Train Loss: 0.2154 | Acc: 0.9500 | Prec: 0.9566 | Rec: 0.8929 | F1: 0.9236 | AUC: 0.9851
  Test   Loss: 0.5931 | Acc: 0.8060 | Prec: 0.8130 | Rec: 0.5556 | F1: 0.6601 | AUC: 0.8063
  Padchest   Loss: 0.4583 | Acc: 0.8472 | Prec: 0.6232 | Rec: 0.4574 | F1: 0.5276 | AUC: 0.7558
  JSRT   Loss: 1.6590 | Acc: 0.5833 | Prec: 0.9375 | Rec: 0.3409 | F1: 0.5000 | AUC: 0.7005
  ✓ Saved new best model! (F1: 0.6601)
Epoch 3/20
  Train Loss: 0.1016 | Acc: 0.9669 | Prec: 0.9459 | Rec: 0.9571 | F1: 0.9515 | AUC: 0.9911
  Test   Loss: 0.8349 | Acc: 0.7985 | Prec: 0.8067 | Rec

In [30]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'rad_cxr14_cosine_{process}_{run_index}_best_classifier_f1_3.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.1119 | Acc: 0.9653 | Prec: 0.9499 | Rec: 0.9476 | F1: 0.9487 | AUC: 0.9881
  Test   Loss: 0.7460 | Acc: 0.8060 | Prec: 0.7895 | Rec: 0.5833 | F1: 0.6709 | AUC: 0.8097
  Padchest   Loss: 0.5969 | Acc: 0.8413 | Prec: 0.5875 | Rec: 0.5000 | F1: 0.5402 | AUC: 0.7510
  JSRT   Loss: 1.8621 | Acc: 0.5833 | Prec: 0.9375 | Rec: 0.3409 | F1: 0.5000 | AUC: 0.7029
  ✓ Saved new best model! (F1: 0.6709)
Epoch 2/20
  Train Loss: 0.1124 | Acc: 0.9669 | Prec: 0.9480 | Rec: 0.9548 | F1: 0.9514 | AUC: 0.9866
  Test   Loss: 0.7366 | Acc: 0.8041 | Prec: 0.7923 | Rec: 0.5722 | F1: 0.6645 | AUC: 0.8070
  Padchest   Loss: 0.5924 | Acc: 0.8373 | Prec: 0.5750 | Rec: 0.4894 | F1: 0.5287 | AUC: 0.7540
  JSRT   Loss: 2.0950 | Acc: 0.5833 | Prec: 0.9375 | Rec: 0.3409 | F1: 0.5000 | AUC: 0.6972
Epoch 3/20
  Train Loss: 0.0823 | Acc: 0.9701 | Prec: 0.9570 | Rec: 0.9548 | F1: 0.9559 | AUC: 0.9944
  Test   Loss: 0.7577 | Acc: 0.8041 | Prec: 0.7969 | Rec: 0.5667 | F1: 0.6623 | AUC: 0.8069
  P